In [1]:
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from actionstudio.src.foundation_modeling.utils.common import *

/fsx/home/jianguozhang/miniconda3/envs/actionstudio/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-0528-Qwen3-8B")
print(tokenizer.get_added_vocab())

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


{'<｜begin▁of▁sentence｜>': 151643, '<|im_start|>': 151644, '<｜end▁of▁sentence｜>': 151645, '<|object_ref_start|>': 151646, '<|object_ref_end|>': 151647, '<|box_start|>': 151648, '<|box_end|>': 151649, '<|quad_start|>': 151650, '<|quad_end|>': 151651, '<|vision_start|>': 151652, '<|vision_end|>': 151653, '<|vision_pad|>': 151654, '<|image_pad|>': 151655, '<|video_pad|>': 151656, '<tool_call>': 151657, '</tool_call>': 151658, '<|fim_prefix|>': 151659, '<|fim_middle|>': 151660, '<|fim_suffix|>': 151661, '<|fim_pad|>': 151662, '<|repo_name|>': 151663, '<|file_sep|>': 151664, '<tool_response>': 151665, '</tool_response>': 151666, '<think>': 151667, '</think>': 151668, '<｜User｜>': 151669, '<｜Assistant｜>': 151670}


In [6]:
tokenizer = AutoTokenizer.from_pretrained("/fsx/home/jianguozhang/checkpoints/qwen3/raw/qwen3_4b_instruct_2507")

# chat_template_name = "qwen3-4b-instruct-2507.jinja"
# chat_template_name = "qwen3-4b-instruct-2507--xlam-nothink.jinja"
chat_template_name = "qwen3-4b-instruct-2507--xlam.jinja"
# chat_template_name = "qwen3-4b-thinking-2507.jinja"
customized_chat_template = open("/fsx/home/jianguozhang/jianguozhang/agentstudio/agentstudio/projects/scripts/jobs/20260128/" + chat_template_name).read()
print(customized_chat_template)
tokenizer.chat_template = customized_chat_template

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}

    {{- "# Tools\n\nYou are a helpful assistant that can use tools. You are developed by Salesforce xLAM team.\n" }}
    {{- "Behavior:\n" }}
    {{- "- You may call one or more functions to assist with the user query.\n" }}
    {{- "- If you decide to call a tool, do not provide a final answer until the tool results are returned. Once results arrive, process them and continue (making further calls if needed).\n" }}
    {{- "- If no tool is suitable, state that explicitly. You may also suggest what tool would be needed, or otherwise respond in plain text or ask for clarification.\n" }}
    {{- "- If the user's input lacks required parameters, ask for clarification before calling any tool.\n" }}
    {{- "- You may make multiple tool calls (sequentially or in parallel) within the same turn if needed.\n" }}
    {{- "- If a tool errors, 

In [20]:
model_path = "/fsx/home/jianguozhang/checkpoints/qwen3/sft/20260128/qwen3_30b_a3b_instruct_2507__20260128_xlam_2_filtered_v4__unify_sft_full_training_seq_len_16384_lr_5e-6_bs_1_ga_3_steps_8943_pre_bf16_model_id__xlam-moe-30b--1/final_merged_checkpoint_torch"
model_path = "/fsx/home/jianguozhang/checkpoints/miles/xlam-qwen3-4b-instruct-2507--1-hf"
model_path = "/fsx/home/jianguozhang/checkpoints/qwen3/raw/qwen3_4b_instruct_2507"
tokenizer_2 = AutoTokenizer.from_pretrained(model_path)
model_2 = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)


Loading checkpoint shards: 100%|██████████| 3/3 [00:41<00:00, 13.78s/it]


In [1]:
messages = [
    {"role": "user", "content": "Hello, what is the weather in palo alto and what are the recommended tourisms in San Francisco?"}
    # {"role": "user", "content": "Hello, what are the recommended tourisms in San Francisco?"}
]
tools = [
  {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "get weather of a location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "location"
                }
            },
            "required": [
                "location"
            ]
        }
    }
},
{
    "type": "function",
    "function": {
        "name": "get_tourisms",
        "description": "get recommended tourisms of a location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "location"   
                }
            },
            "required": [
                "location"
            ]
        }
    }
}
]
prompt =  tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True
    )

print(prompt)

NameError: name 'tokenizer' is not defined

In [ ]:
model_inputs = tokenizer([prompt], return_tensors="pt").to(model_2.device)

# conduct text completion
generated_ids = model_2.generate(
    **model_inputs,
    max_new_tokens=16384,
    temperature=0.001
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = tokenizer.decode(output_ids, skip_special_tokens=True)

print("content:\n", content)

content:
 <tool_call>
{"name": "get_weather", "arguments": {"location": "Palo Alto"}}
</tool_call>
<tool_call>
{"name": "get_tourisms", "arguments": {"location": "San Francisco"}}
</tool_call>


: 

In [ ]:
temp=[{"name": "get_weather", "arguments": {"location": "Palo Alto"}}]
print(json.dumps(temp, indent=4))

[
    {
        "name": "get_weather",
        "arguments": {
            "location": "Palo Alto"
        }
    }
]


: 

: 

In [ ]:
prompt

'<|im_start|>system\n# Tools\n\nYou are a helpful assistant that can use tools. You are developed by Salesforce xLAM team.\nBehavior:\n- You may call one or more functions to assist with the user query.\n- If you decide to call a tool, do not provide a final answer until the tool results are returned. Once results arrive, process them and continue (making further calls if needed).\n- If no tool is suitable, state that explicitly. You may also suggest what tool would be needed, or otherwise respond in plain text or ask for clarification.\n- If the user\'s input lacks required parameters, ask for clarification before calling any tool.\n- You may make multiple tool calls (sequentially or in parallel) within the same turn if needed.\n- If a tool errors, handle gracefully: retry if appropriate, or briefly explain and proceed (or ask for clarification).\n- If no tool is needed (e.g., casual conversation or general advice), respond directly in plain text.\n\nYou are provided with function sig

In [ ]:
prompt==prompt_1

True

In [35]:
tokenizer_file = open_json("/fsx/home/jianguozhang/checkpoints/qwen3/sft/20260128/qwen3_30b_a3b_instruct_2507__20260128_xlam_2_filtered_v4__unify_sft_full_training_seq_len_16384_lr_5e-6_bs_1_ga_3_steps_8943_pre_bf16_model_id__xlam-moe-30b--1/final_merged_checkpoint_torch/tokenizer_config.json")
print(tokenizer_file["chat_template"])

KeyError: 'chat_template'

In [10]:
temp="{%- if tools %}\n    {{- '<|im_start|>system\\n' }}\n    {%- if messages[0].role == 'system' %}\n        {{- messages[0].content + '\\n\\n' }}\n    {%- endif %}\n    {{- \"# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>\" }}\n    {%- for tool in tools %}\n        {{- \"\\n\" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- \"\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\\"name\\\": <function-name>, \\\"arguments\\\": <args-json-object>}\\n</tool_call><|im_end|>\\n\" }}\n{%- else %}\n    {%- if messages[0].role == 'system' %}\n        {{- '<|im_start|>system\\n' + messages[0].content + '<|im_end|>\\n' }}\n    {%- endif %}\n{%- endif %}\n{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}\n{%- for message in messages[::-1] %}\n    {%- set index = (messages|length - 1) - loop.index0 %}\n    {%- if ns.multi_step_tool and message.role == \"user\" and message.content is string and not(message.content.startswith('<tool_response>') and message.content.endswith('</tool_response>')) %}\n        {%- set ns.multi_step_tool = false %}\n        {%- set ns.last_query_index = index %}\n    {%- endif %}\n{%- endfor %}\n{%- for message in messages %}\n    {%- if message.content is string %}\n        {%- set content = message.content %}\n    {%- else %}\n        {%- set content = '' %}\n    {%- endif %}\n    {%- if (message.role == \"user\") or (message.role == \"system\" and not loop.first) %}\n        {{- '<|im_start|>' + message.role + '\\n' + content + '<|im_end|>' + '\\n' }}\n    {%- elif message.role == \"assistant\" %}\n        {%- set reasoning_content = '' %}\n        {%- if message.reasoning_content is string %}\n            {%- set reasoning_content = message.reasoning_content %}\n        {%- else %}\n            {%- if '</think>' in content %}\n                {%- set reasoning_content = content.split('</think>')[0].rstrip('\\n').split('<think>')[-1].lstrip('\\n') %}\n                {%- set content = content.split('</think>')[-1].lstrip('\\n') %}\n            {%- endif %}\n        {%- endif %}\n        {%- if loop.index0 > ns.last_query_index %}\n            {%- if loop.last or (not loop.last and reasoning_content) %}\n                {{- '<|im_start|>' + message.role + '\\n<think>\\n' + reasoning_content.strip('\\n') + '\\n</think>\\n\\n' + content.lstrip('\\n') }}\n            {%- else %}\n                {{- '<|im_start|>' + message.role + '\\n' + content }}\n            {%- endif %}\n        {%- else %}\n            {{- '<|im_start|>' + message.role + '\\n' + content }}\n        {%- endif %}\n        {%- if message.tool_calls %}\n            {%- for tool_call in message.tool_calls %}\n                {%- if (loop.first and content) or (not loop.first) %}\n                    {{- '\\n' }}\n                {%- endif %}\n                {%- if tool_call.function %}\n                    {%- set tool_call = tool_call.function %}\n                {%- endif %}\n                {{- '<tool_call>\\n{\"name\": \"' }}\n                {{- tool_call.name }}\n                {{- '\", \"arguments\": ' }}\n                {%- if tool_call.arguments is string %}\n                    {{- tool_call.arguments }}\n                {%- else %}\n                    {{- tool_call.arguments | tojson }}\n                {%- endif %}\n                {{- '}\\n</tool_call>' }}\n            {%- endfor %}\n        {%- endif %}\n        {{- '<|im_end|>\\n' }}\n    {%- elif message.role == \"tool\" %}\n        {%- if loop.first or (messages[loop.index0 - 1].role != \"tool\") %}\n            {{- '<|im_start|>user' }}\n        {%- endif %}\n        {{- '\\n<tool_response>\\n' }}\n        {{- content }}\n        {{- '\\n</tool_response>' }}\n        {%- if loop.last or (messages[loop.index0 + 1].role != \"tool\") %}\n            {{- '<|im_end|>\\n' }}\n        {%- endif %}\n    {%- endif %}\n{%- endfor %}\n{%- if add_generation_prompt %}\n    {{- '<|im_start|>assistant\\n<think>\\n' }}\n{%- endif %}"

temp == customized_chat_template

True

In [ ]:
messages = [
    {"role": "user", "content": "Hello, how are you, what is the current time?"},
    {
        "role": "assistant",
        "content": "Sure, I can help you to <think>YANMEI\nHELLO</think> check it.",
        # "reasoning_content": "I will </think>JIANGUO ZHANG\n</think> and the available stocks.",
        "tool_calls": [
            {
                "type": "function",
                "function": {
                    "name": "get_current_time",
                    "arguments": "{}"
                },
                "id": "633159740"
            },
            {
                "type": "function",
                "function": {
                    "name": "get_available_stocks",
                    "arguments": "{\"sector\": \"Technology\"}"
                },
                "id": "294116824"
            }
        ]
    },
    {
        "role": "tool",
        "content": "{\"current_time\": \"10:30 AM\"}",
        "tool_call_id": "633159740"
    },
    {
        "role": "tool",
        "content": "{\"stock_list\": [\"AAPL\", \"GOOG\", \"MSFT\", \"NVDA\", \"AAPL\", \"GOOG\", \"MSFT\", \"NVDA\", \"LGY\", \"HW\", \"OLMK\", \"O\", \"ZTMG\", \"BZHAI\", \"CEXPE\", \"MQZRF\", \"XBKC\", \"W\", \"RHWL\", \"CL\", \"KO\", \"WG\", \"ST\", \"RNUUI\", \"NFLP\", \"LBCV\", \"U\", \"Z\", \"ABBBQ\", \"QCZHX\", \"TMMOX\", \"STAIK\", \"GGSSJ\", \"M\", \"BMXI\", \"IY\", \"DAJ\", \"DV\", \"JCVLY\", \"TPE\", \"CJBVY\", \"OU\", \"TMH\", \"USYPG\", \"DOXG\", \"SCT\", \"DIZE\", \"X\", \"G\", \"MZD\", \"ZILO\", \"P\", \"LLA\", \"QUGWR\", \"LBC\", \"STI\", \"WDQ\", \"EH\", \"YF\", \"TBYI\", \"ROV\", \"DB\", \"J\", \"OWN\", \"P\", \"UHDC\", \"FMCE\", \"EJ\", \"YEFP\", \"YKT\", \"TJGK\", \"AJ\", \"YT\", \"JFMA\", \"CUA\", \"V\", \"ID\", \"SBSXN\", \"UUZTD\", \"TNI\", \"T\", \"O\", \"EV\", \"VVCG\", \"UP\", \"T\", \"XGED\", \"RVXY\", \"P\", \"PWDZ\", \"LJS\", \"QM\", \"ILZ\", \"RET\", \"J\", \"T\", \"IIH\", \"Z\", \"ONUWE\", \"PX\", \"LYV\", \"BXNA\", \"BVSS\", \"Z\", \"NVRLW\", \"OKO\", \"RHH\", \"O\", \"GGWPW\", \"I\", \"BGIT\", \"EC\", \"WAGD\", \"R\", \"E\", \"ZK\", \"KW\", \"PQ\", \"Z\", \"XW\", \"IZS\", \"G\", \"ZMROV\", \"N\", \"ILW\", \"TN\", \"RKV\", \"LBB\", \"LXE\", \"TWB\", \"FLQBX\", \"OLXN\", \"KPC\", \"J\", \"YA\", \"SUTO\", \"BAFYF\", \"HLCNY\", \"ZS\", \"OB\", \"MURJK\", \"OI\", \"U\", \"QZEO\", \"QNR\", \"DLE\", \"SREFX\", \"YNN\", \"ZT\", \"I\", \"GMET\", \"CF\", \"IJI\", \"A\", \"IJJE\", \"IIPE\", \"FQV\", \"ZHUN\", \"ZNQSI\", \"A\", \"AAOOS\", \"C\", \"XXAET\", \"LG\", \"BYJAN\", \"USZRZ\", \"ORATF\", \"WW\", \"PTH\", \"BDLGN\", \"NA\", \"UQ\", \"IOJJO\", \"JBTPW\", \"B\", \"J\", \"SEM\", \"XJP\", \"H\", \"PO\", \"JX\", \"XBW\", \"GZKLB\", \"QMXLV\", \"GST\", \"OMMPT\", \"V\", \"P\", \"GP\", \"J\", \"AKSGL\", \"HUXV\", \"AHV\", \"CXVEI\", \"UCMDX\", \"X\", \"NVU\", \"SJ\", \"QM\", \"ZRK\", \"SB\", \"SIZ\", \"QHH\", \"PDYP\", \"JTKS\", \"V\", \"PPE\", \"RVB\", \"XMGSL\", \"N\", \"JE\", \"VMUH\", \"ZRE\", \"W\", \"Q\", \"XWC\", \"VXOD\", \"ESQ\", \"PXX\", \"BS\", \"FWW\", \"TAK\", \"A\", \"L\", \"T\", \"Q\", \"LIWK\", \"X\", \"K\", \"LYH\", \"EKDP\", \"P\", \"Y\", \"IULD\", \"P\", \"TA\", \"JRT\", \"OL\", \"QQIJY\", \"DO\", \"QLT\", \"COBXD\", \"BXS\", \"B\", \"FOCT\", \"K\", \"CPOIZ\", \"I\", \"AWMC\", \"WVABM\", \"X\", \"ZYSC\", \"EROQ\", \"D\", \"SMIO\", \"XS\", \"AOXBR\", \"UOLK\", \"D\", \"M\", \"YZDXS\", \"OYTFO\", \"RVKBA\", \"ZH\", \"I\", \"GRE\", \"WHK\", \"UBJ\", \"V\", \"LX\", \"O\", \"UK\", \"JBQF\", \"WAWUE\", \"Z\", \"IUWPI\", \"ZV\", \"E\", \"DWPFL\", \"D\", \"KLBMK\", \"YMU\", \"QANOX\", \"EEMGR\", \"EDICO\", \"MQ\", \"A\", \"CYI\", \"JGWF\", \"E\", \"SE\", \"A\", \"KHYM\", \"G\", \"SROM\", \"YNJKH\", \"PMUN\", \"GNIKB\", \"JH\", \"O\", \"LVYRM\", \"KLN\", \"H\", \"VF\", \"SBTTV\", \"XNRZJ\", \"MMR\", \"QVBIX\", \"NYGEY\", \"ZTI\", \"UUB\", \"P\", \"MQBB\", \"ESJNT\", \"OW\", \"XNS\", \"XWFI\", \"LQ\", \"AOXC\", \"EUUF\", \"ADTMM\", \"GQ\", \"PGL\", \"ZOMAS\", \"NUB\", \"FDO\", \"ZN\", \"CTVV\", \"XFP\", \"AIQI\", \"KPULF\", \"PEGNC\", \"DVQC\", \"EUNR\", \"LMDB\", \"TWI\", \"MQO\", \"U\", \"NLJJ\", \"IK\", \"KCUZ\", \"ZY\", \"GNZ\", \"DVEHH\", \"AAJ\", \"VJYJY\", \"R\", \"RUQFZ\", \"VLZ\", \"IH\", \"BI\", \"L\", \"T\", \"F\", \"BR\", \"D\", \"QZJHI\", \"JH\", \"OO\", \"R\", \"BZM\", \"EY\", \"KUA\", \"Q\", \"GIE\", \"HK\", \"FK\", \"TDS\", \"BLYFT\", \"CZY\", \"NPN\", \"O\", \"YOLCR\", \"GPTTY\", \"N\", \"JSOM\", \"ODB\", \"BOF\", \"RPA\", \"TYUE\", \"YJXQC\", \"SAZ\", \"UOI\", \"R\", \"BLJM\", \"UCJKW\", \"CPHDJ\", \"QB\", \"ZZB\", \"Q\", \"QFNE\", \"ZHFOF\", \"EJQ\", \"RZX\", \"CS\", \"DDNYC\", \"SLT\", \"LE\", \"UV\", \"FS\", \"GP\", \"U\", \"M\", \"TDG\", \"U\", \"MWAP\", \"YTR\", \"X\", \"GWLU\", \"M\", \"Y\", \"IX\", \"VDV\", \"TISAC\", \"L\", \"F\", \"T\", \"ZNS\", \"EXX\", \"I\", \"SBGS\", \"BWT\", \"XNUKY\", \"W\", \"UCEU\", \"B\", \"S\", \"WKFR\", \"FLLRO\", \"BVTR\", \"FNAJ\", \"ML\", \"MG\", \"QJ\", \"LDCTR\", \"THIV\", \"HIM\", \"HRC\", \"L\", \"ROX\", \"O\", \"UKP\", \"YHG\", \"QNGQS\", \"SUJ\", \"YWMW\", \"LNWDJ\", \"YTH\", \"DGJFJ\", \"BGCU\", \"ADUH\", \"KETP\", \"Y\", \"EQ\", \"NHZUS\", \"PJ\", \"GGUMD\", \"NAHZW\", \"SSU\", \"O\", \"U\", \"T\", \"XXR\", \"DTVSN\", \"IST\", \"KDLFP\", \"QFD\", \"DHTT\", \"YT\", \"PXIU\", \"ZISP\", \"GEKN\", \"PC\", \"FPVGA\", \"QCKYQ\", \"QYSCY\", \"X\", \"JAMYJ\", \"FGDY\", \"U\", \"ZIUU\", \"WDXK\", \"UXE\", \"RP\", \"KMTFP\", \"E\", \"QQAWD\", \"C\", \"X\", \"QUJ\", \"SONRZ\", \"N\", \"LRF\", \"P\", \"FF\", \"ZPCDE\", \"QFC\", \"XYVFP\", \"VQVVN\", \"P\", \"RXMQ\", \"VLFCQ\", \"SYW\", \"RJSB\", \"JHBGM\", \"VFELK\", \"Y\", \"ETR\", \"NAJIV\", \"C\", \"G\", \"ABDW\", \"LG\", \"AANNY\", \"ZFED\", \"LT\", \"C\", \"RHYU\", \"PBJRL\", \"PVH\", \"GZDHN\", \"O\", \"D\", \"SR\", \"PC\", \"A\", \"NVJ\", \"VCF\", \"UN\", \"ZWRO\", \"LJD\", \"EA\", \"Y\", \"ZE\", \"EQHBL\", \"YNH\", \"VHE\", \"BFTR\", \"CIXYV\", \"M\", \"BNX\", \"J\", \"T\", \"ZFCZ\", \"KNBXP\", \"K\", \"V\", \"CV\", \"F\", \"GJUI\", \"JHWCR\", \"P\", \"ACVFL\", \"BAWOD\", \"Z\", \"PP\", \"MJ\", \"ICV\", \"ZNWEK\", \"F\", \"L\", \"WMGE\", \"GMGBV\", \"UGA\", \"L\", \"V\", \"UYC\", \"UXYA\", \"S\", \"IYTO\", \"UV\", \"V\", \"TTAP\", \"XUZ\", \"WHQO\", \"LYPLL\", \"TZIP\", \"Z\", \"BC\", \"GTLSX\", \"R\", \"T\", \"U\", \"GCQ\", \"BI\", \"JGSW\", \"WE\", \"MZGM\", \"MB\", \"TYMTA\", \"LDWU\", \"QLF\", \"H\", \"SPVTF\", \"HK\", \"LA\", \"KJA\", \"Q\", \"RZVEN\", \"LUR\", \"CZZMC\", \"X\", \"WPA\", \"ES\", \"BJ\", \"HSUN\", \"UNMH\", \"N\", \"ZYRH\", \"YR\", \"IRR\", \"ZRQH\", \"MNJD\", \"Y\", \"N\", \"KP\", \"T\", \"AC\", \"GADM\", \"GLCW\", \"WZIII\", \"XRD\", \"XK\", \"EQOQ\", \"QI\", \"S\", \"MTRK\", \"R\", \"BEPDR\", \"NMRRK\", \"BUO\", \"L\", \"P\", \"XJEY\", \"GULRJ\", \"GKCIO\", \"OOON\", \"DD\", \"EKUPJ\", \"BORN\", \"UNGHF\", \"QCN\", \"ZHQN\", \"IPZ\", \"BKGUA\", \"V\", \"OE\", \"LUKZN\", \"KEOOG\", \"LOAE\", \"VPRG\", \"CA\", \"EBJIH\", \"UTPL\", \"WNTI\", \"WS\", \"PL\", \"QGXRE\", \"ZF\", \"CZ\", \"LJCWF\", \"FFGN\", \"ZVE\", \"NQ\", \"GWXZQ\", \"JGFI\", \"COAQS\", \"ZUZOQ\", \"B\", \"MIYPA\", \"RJ\", \"SLC\", \"TOMA\", \"DSTRT\", \"VU\", \"JZ\", \"BAS\", \"VKWN\", \"YPPFU\", \"KNN\", \"YOZKD\", \"F\", \"VWWEI\", \"A\", \"SVIT\", \"F\", \"XZTT\", \"P\", \"DJUD\", \"OVP\", \"IMCIE\", \"P\", \"YO\", \"SQUB\", \"KXHVG\", \"XJP\", \"W\", \"O\", \"LC\", \"UQ\", \"MX\", \"EG\", \"NOB\", \"M\", \"RCFSJ\", \"V\", \"LEJ\", \"R\", \"MZJTS\", \"D\", \"GWO\", \"KOUHA\", \"C\", \"MGSQ\", \"CAAMJ\", \"ZFBA\", \"HJY\", \"R\", \"WWSZL\", \"TVYW\", \"MIHZE\", \"ZM\", \"CS\", \"PWU\", \"PUFZ\", \"AOU\", \"X\", \"GZP\", \"N\", \"TCMGE\", \"LZCNC\", \"BRIP\", \"TZ\", \"CLI\", \"VHLR\", \"D\", \"CPOYH\", \"ZJGRO\", \"LWUZ\", \"QZQYC\", \"UHMR\", \"ILK\", \"KMSEE\", \"G\", \"JTXY\", \"S\", \"E\", \"RO\", \"UAXD\", \"WGYYZ\", \"UG\", \"QD\", \"YIVP\", \"J\", \"LMA\", \"DLPH\", \"H\", \"VKA\", \"DBJ\", \"MHKE\", \"TV\", \"WB\", \"SP\", \"OSEC\", \"RZ\", \"BJG\", \"CGI\", \"GPUHT\", \"OJ\", \"SBFFJ\", \"BOPLK\", \"R\", \"PA\", \"OOQ\", \"E\", \"OFI\", \"GFRT\", \"UYIA\", \"PXGKE\", \"WWSG\", \"B\", \"ZSCK\", \"RJGL\", \"DIC\", \"GK\", \"PWPF\", \"DF\", \"K\", \"UR\", \"HY\", \"AA\", \"IABI\", \"S\", \"N\", \"QIEW\", \"RLHQD\", \"ZVXJJ\", \"YEO\", \"WDWZ\", \"GJ\", \"CYOA\", \"U\", \"V\", \"QKN\", \"V\", \"MY\", \"JMMZA\", \"FXN\", \"RJW\", \"U\", \"NGATS\", \"YOJRT\", \"QLFUU\", \"QRO\", \"YU\", \"NP\", \"IOYQX\", \"XX\", \"TJQLA\", \"VSEPP\", \"QJ\", \"ZNZYC\", \"TT\", \"OBHED\", \"UEM\", \"WFTF\", \"EBDA\", \"IY\", \"PF\", \"RGXTC\", \"SMWL\", \"HF\", \"T\", \"CRDG\", \"MBXF\", \"AVZNI\", \"ERUY\", \"DFAD\", \"YE\", \"PGT\", \"UNJH\", \"UNR\", \"K\", \"VLBYA\", \"CLD\", \"FSRAV\", \"IAW\", \"INM\", \"LUPTK\", \"G\", \"ZXCAN\", \"EMY\", \"L\", \"VGP\", \"AY\", \"ZUT\", \"NVUI\", \"SZ\", \"FTI\", \"WF\", \"A\", \"BHD\", \"MWRYL\", \"VVNV\", \"JYVJ\", \"OSZQN\", \"SPUZH\", \"U\", \"VF\", \"GXYNK\", \"LXSM\", \"GLY\", \"AJYW\", \"XZ\", \"E\", \"X\", \"MGUV\", \"KKM\", \"JJY\", \"JDIOB\", \"BITQ\", \"ABV\", \"MWRWZ\", \"OGCLJ\", \"OAH\", \"H\", \"B\", \"PH\", \"C\", \"BUBJ\", \"XAV\", \"BAPUB\", \"L\", \"IVM\", \"AGK\", \"HEDQC\", \"BJPA\", \"L\", \"GR\", \"NLVIA\", \"VQTS\", \"VAE\", \"VQ\", \"DBY\", \"EYRZ\", \"RIKW\", \"R\", \"BKNSX\", \"PRNN\", \"JVD\", \"R\", \"Z\", \"BCSOX\", \"JRT\", \"BFJ\", \"S\", \"QV\", \"VVIUC\", \"W\", \"RNJIG\", \"GDQMC\", \"VRD\", \"B\", \"TBP\", \"RF\", \"IKB\", \"KURKI\", \"GF\", \"KFIG\", \"KIP\", \"FZER\", \"UZVPL\", \"TFCQ\", \"NDXQF\", \"QOU\", \"FP\", \"R\", \"GKQZT\", \"Z\", \"S\", \"YBRWT\", \"AIL\", \"NJEJU\", \"NF\", \"RXBP\", \"RPD\", \"XPQGJ\", \"V\", \"XCQF\", \"POVOI\", \"XOSKL\", \"O\", \"VGU\", \"HZ\", \"BQG\", \"CKPG\", \"ZC\", \"SOMX\", \"IHQL\", \"FPW\", \"DLOSK\", \"WEEY\", \"YQ\", \"XRYBN\", \"JBB\", \"AT\", \"V\", \"XXY\", \"S\", \"QJT\", \"AUL\", \"TPEP\", \"DGRS\", \"AU\", \"QB\", \"CT\", \"WND\", \"VEM\", \"F\", \"S\", \"XL\", \"XVGD\", \"UYAS\", \"X\", \"YDJM\", \"FVOWH\", \"YT\", \"LCQR\", \"F\", \"FID\", \"MJKQ\", \"R\", \"ZLF\", \"LS\", \"P\", \"GG\", \"PE\", \"CHAC\", \"MMIUH\", \"AFDLK\", \"XS\", \"BMY\", \"C\", \"FDVU\"]}",
        "tool_call_id": "294116824"
    },
    {"role": "assistant", "content": "The current time is 2026-01-29 10:00:00"},
    {"role": "user", "content": "What is the capital of the moon?"},
    {
        "role": "assistant", 
        "content": "The capital of the moon is <think>YANMEI\nHELLO</think> the moon!",
        # "content": "The capital of the moon is the moon!",
        # "reasoning_content": "JIANGUO ZHANG\nHELLO",
        "tool_calls": [
            {
                "type": "function",
                "function": {"name": "get_current_time", "arguments": "{}"}
            }
        ]
    },
]
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "arguments": "{}"},
        "id": "633159740"
    },
    {
        "type": "function",
        "function": {
            "name": "get_available_stocks",
            "arguments": "{\"sector\": \"Technology\"}"
        },
        "id": "294116824"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True
    )

print(prompt)


<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_current_time", "arguments": "{}"}, "id": "633159740"}
{"type": "function", "function": {"name": "get_available_stocks", "arguments": "{\"sector\": \"Technology\"}"}, "id": "294116824"}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Hello, how are you, what is the current time?<|im_end|>
<|im_start|>assistant
 check it.
<tool_call>
{"name": "get_current_time", "arguments": {}}
</tool_call>
<tool_call>
{"name": "get_available_stocks", "arguments": {"sector": "Technology"}}
</tool_call><|im_end|>
<|im_start|>user
<tool_response>
{"current_time": "10:30 AM"}
</tool_response>
<too

: 

In [ ]:

def gen_multi_turn_loss_mask_qwen3(
        self, messages: list[dict], tools: list[dict] = None
    ) -> tuple[list[int], list[int]]:
        all_loss_masks = []
        all_token_ids = []

        prefix_message = {"role": "user", "content": "FOR CALCULATING LOSS MASK ONLY"}
        prefix_token_ids = self.tokenizer.apply_chat_template([prefix_message], tokenize=True)

        for i, message in enumerate(messages):
            if i == 0:
                tailed_message_ids = self.tokenizer.apply_chat_template(
                    [message, prefix_message], tokenize=True, tools=tools
                )
                message_ids = tailed_message_ids[: -len(prefix_token_ids)]
            else:
                prefixed_message_ids = self.tokenizer.apply_chat_template([prefix_message, message], tokenize=True)
                message_ids = prefixed_message_ids[len(prefix_token_ids) :]

            if message["role"] != "system" and i > 0:
                message_ids = message_ids[self.system_message_length :]

            if message["role"] == "assistant":
                loss_mask = [0] * self.gen_token_length + [1] * (len(message_ids) - self.gen_token_length)
            else:
                loss_mask = [0] * len(message_ids)

            if message.get("step_loss_mask", 1) != 1:
                loss_mask = [0] * len(message_ids)

            all_loss_masks.extend(loss_mask)
            all_token_ids.extend(message_ids)

        return all_token_ids, all_loss_masks

<|im_start|>system
You are a helpful assistant. You are developed by Salesforce xLAM team.<|im_end|>
<|im_start|>user
Hello, how are you, what is the current time?<|im_end|>
<|im_start|>assistant
Sure, I can help you to check it.
<tool_call>
{"name": "get_current_time", "arguments": {}}
</tool_call>
<tool_call>
{"name": "get_available_stocks", "arguments": {"sector": "Technology"}}
</tool_call><|im_end|>
<|im_start|>tool
<tool_response>
{"current_time": "10:30 AM"}
</tool_response>
<tool_response>
{"stock_list": ["AAPL", "GOOG", "MSFT", "NVDA", "AAPL", "GOOG", "MSFT", "NVDA", "LGY", "HW", "OLMK", "O", "ZTMG", "BZHAI", "CEXPE", "MQZRF", "XBKC", "W", "RHWL", "CL", "KO", "WG", "ST", "RNUUI", "NFLP", "LBCV", "U", "Z", "ABBBQ", "QCZHX", "TMMOX", "STAIK", "GGSSJ", "M", "BMXI", "IY", "DAJ", "DV", "JCVLY", "TPE", "CJBVY", "OU", "TMH", "USYPG", "DOXG", "SCT", "DIZE", "X", "G", "MZD", "ZILO", "P", "LLA", "QUGWR", "LBC", "STI", "WDQ", "EH", "YF", "TBYI", "ROV", "DB", "J", "OWN", "P", "UHDC", "FMC

In [38]:
prompt==prompt_2

True

In [52]:
model_name = "qwen3_30b_a3b_instruct_2507"
model_name = "qwen3_235b_a22b_instruct_2507"
model_name = "qwen3_4b_instruct_2507"
tokenizer_file = open_json("/fsx/home/jianguozhang/checkpoints/qwen3/raw/" + model_name + "/tokenizer_config.json")
customized_chat_template = open("/fsx/home/jianguozhang/jianguozhang/agentstudio/agentstudio/projects/scripts/jobs/20260128/qwen3-4b-instruct-2507--xlam-nothink.jinja").read()
tokenizer_file["chat_template"] = customized_chat_template
save_json("/fsx/home/jianguozhang/checkpoints/qwen3/raw/" + model_name + "/tokenizer_config.json", tokenizer_file)
print("done for model: ", model_name)

done for model:  qwen3_4b_instruct_2507


In [5]:
model_name = "qwen3_235b_a22b_instruct_2507"
model_name = "qwen3_30b_a3b_instruct_2507"
model_name = "qwen3_4b_instruct_2507"
model_name = "qwen3_4b_thinking_2507"
model_name = "qwen3_30b_a3b_thinking_2507"
model_name = "qwen3_235b_a22b_thinking_2507"
tokenizer_file = open_json("/fsx/home/jianguozhang/checkpoints/qwen3/raw/" + model_name + "/tokenizer_config.json")
print(tokenizer_file["chat_template"])

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}
{%- for message in messages[::-1] %}
    {%- set index = (messages|length - 

In [3]:
import random
models = [
    # "Salesforce/xLAM-2-3b-fc-r",
    "Salesforce/Llama-xLAM-2-8b-fc-r",
    "Salesforce/xLAM-7b-r",
    "Salesforce/xLAM-2-1b-fc-r"
]

for model in models:
    for idx in range(random.randint(500, 1500)):
        auto_tokenizer = AutoTokenizer.from_pretrained(model)
        auto_config = AutoConfig.from_pretrained(model)
    print("done for model: ", model)

done for model:  Salesforce/Llama-xLAM-2-8b-fc-r
done for model:  Salesforce/xLAM-7b-r
done for model:  Salesforce/xLAM-2-1b-fc-r
